# Step 3 · Odds from Kalshi

**What this notebook does:** downloads Kalshi's current prices for "Nobel Prize in Economics 2026" and turns
them into a chance for each person. Saves `kalshi_odds.csv`.

**About Kalshi.** Kalshi is a regulated US prediction market. For each name there is a yes/no contract that
pays $1 if that person wins. If "yes" costs 13 cents, traders put the chance at about 13%.
A prize can be shared by up to three people, so the chances can add up to more than 100%.

**The API.** Kalshi publishes its prices through a public API. Reading prices needs no account or key
(keys are only needed for trading). Kalshi's code for the Economics Nobel is `KXNOBELECON`; the 2026 prize
is the event `KXNOBELECON-26`, with one "market" per person.

Run this notebook whenever you want fresh odds. It takes a second.

## Setup

`requests` downloads data from the web, `pandas` makes the table, `datetime` records when we downloaded.

In [1]:
import datetime
import warnings

# The Mac's built-in Python prints a harmless warning about its SSL library when `requests` loads.
# Hide it. (This line has to come before `import requests`.)
warnings.filterwarnings("ignore")

import pandas as pd
import requests

## 1. Download the market

The API answers with JSON: a list of "events" (one per year), and inside each event a list of "markets"
(one per person).

In [2]:
URL = "https://api.elections.kalshi.com/trade-api/v2/events"
params = {
    "series_ticker": "KXNOBELECON",    # Kalshi's code for the Economics Nobel
    "with_nested_markets": "true",     # include each person's market inside its event
}

response = requests.get(URL, params=params, timeout=60)
response.raise_for_status()            # stop here if the download failed
data = response.json()

# Keep the 2026 event.
event_2026 = None
for event in data["events"]:
    if event["event_ticker"] == "KXNOBELECON-26":
        event_2026 = event

print(event_2026["title"], ":", len(event_2026["markets"]), "names")

Nobel Prize in Economics 2026 : 9 names


## 2. One row per person

For each person the market has three prices, in dollars:

- **bid**: the highest price someone is currently willing to pay for "yes",
- **ask**: the lowest price someone is currently willing to sell "yes" for,
- **last**: the price of the most recent trade (0 if there hasn't been one).

This market doesn't trade often, so the last trade can be old or missing. We use the **midpoint** between the
bid and the ask as the chance: with a bid of 9 cents and an ask of 16 cents, the chance is 12.5%, which we show
as 13% (halves are rounded up).

In [3]:
rows = []

for market in event_2026["markets"]:
    name = market["yes_sub_title"]                            # the person this contract is about

    # Prices come as text in dollars, e.g. "0.0900". Turn them into whole cents: 9.
    bid = round(float(market["yes_bid_dollars"]) * 100)
    ask = round(float(market["yes_ask_dollars"]) * 100)
    last = round(float(market["last_price_dollars"]) * 100)

    # Midpoint of bid and ask, halves rounded up. In whole numbers: (9 + 16 + 1) // 2 = 13.
    # (// is division that drops the remainder.)
    chance = (bid + ask + 1) // 2

    rows.append({
        "name": name,
        "chance_pct": chance,
        "bid_cents": bid,
        "ask_cents": ask,
        "last_trade_cents": last,
        "kalshi_ticker": market["ticker"],
    })

kalshi = pd.DataFrame(rows).sort_values("chance_pct", ascending=False)

# Note the time of the download (in UTC, the standard world clock).
kalshi["downloaded_at"] = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
kalshi

,name,chance_pct,bid_cents,ask_cents,last_trade_cents,kalshi_ticker,downloaded_at
7,Ariel Pakes,23,22,23,23,KXNOBELECON-26-ARI,2026-09-26 23:40 UTC
2,Richard Blundell,13,9,16,14,KXNOBELECON-26-RIC,2026-09-26 23:40 UTC
3,Susan Athey,13,9,16,16,KXNOBELECON-26-SUS,2026-09-26 23:40 UTC
0,Robert Barro,11,7,14,0,KXNOBELECON-26-ROB,2026-09-26 23:40 UTC
1,Partha Dasgupta,10,6,13,6,KXNOBELECON-26-PAR,2026-09-26 23:40 UTC
4,Ernst Fehr,10,6,14,0,KXNOBELECON-26-ERN,2026-09-26 23:40 UTC
6,Andreu Mas-Colell,10,6,14,0,KXNOBELECON-26-AND,2026-09-26 23:40 UTC
5,Matthew Rabin,9,5,13,0,KXNOBELECON-26-MAT,2026-09-26 23:40 UTC
8,Luigi Zingales,5,2,8,2,KXNOBELECON-26-LUI,2026-09-26 23:40 UTC


## 3. Save

In [4]:
kalshi.to_csv("kalshi_odds.csv", index=False)
print("Saved", len(kalshi), "names to kalshi_odds.csv")

Saved 9 names to kalshi_odds.csv
